# 02 — Preprocessing: clean node table and outcome labels

**Input (immutable):** `nodes.csv` (raw crawl, 51,486 tracks) and `edges.csv` (62,300 edges, already deduplicated and cleaned into a DAG upstream).
**Output:** `nodes_clean.csv`, with one row per track, uninformative columns dropped, and outcome labels added.

This notebook makes one analytical commitment and records it explicitly:

> **The "was it remixed?" outcome is defined from the reconstructed lineage graph
> (the track has out-degree > 0 in the local remix graph), *not* from the site's
> `n_remixes` counter.**

The two disagree for 2.7 % of tracks (1,387). We use the graph because every graph-positive has a real, dated child track, which the leakage-safe time-based features in `03` require.

The counter and an audit flag (`counter_only_positive`) are kept as columns so the choice is fully auditable. The sensitivity check in `04` shows that excluding the disputed tracks leaves the results unchanged (test AUC 0.842 → 0.843).

*Paths are relative to the project root (`data/raw`, `data/processed`).*

In [1]:
import json
import numpy as np
import pandas as pd

RAW_NODES  = "data/processed/nodes.csv"          # <- your data/processed/nodes.csv
RAW_EDGES  = "data/processed/edges.csv"          # <- your data/processed/edges.csv
SUMMARY    = "data/processed/summary.json"       # <- canonical numbers, used for self-checks
OUT_NODES  = "data/processed/nodes_clean.csv"    # written next to this notebook

nodes = pd.read_csv(RAW_NODES)
edges = pd.read_csv(RAW_EDGES)
summary = json.load(open(SUMMARY))

print("raw nodes:", nodes.shape, "| raw edges:", edges.shape)
print("columns:", list(nodes.columns))

raw nodes: (51486, 21) | raw edges: (62300, 8)
columns: ['upload_id', 'user_name', 'user_id', 'user_real_name', 'name', 'date_iso', 'date_unix', 'year', 'license', 'license_url', 'score', 'num_scores', 'num_playlists', 'thumbs_up', 'num_files', 'n_remixes', 'n_sources', 'usertags', 'systags', 'page_url', 'has_meta']


## 1. Drop columns that carry no usable signal

Each drop is a decision an examiner might probe, so each gets a one-line reason.
Verified against the real rows (not assumed):

| Column | Why dropped |
|---|---|
| `user_id` | 100 % null (not exposed by the graph dataview). `user_name` carries author identity instead. |
| `year` | 100 % null. The real date lives in `date_iso` / `date_unix`. |
| `num_playlists` | 100 % null (only available via a non-bulk endpoint; excluded by design). |
| `score` | 90.1 % null, values 100–500, semantics unclear (looks like a sparse rating, not a count). Not usable. |
| `thumbs_up` | Constant value `1` for every row — zero information. |
| `has_meta` | Constant `False` for every row — flag for the abandoned second metadata pass; never set. |

`num_scores` is **kept** (it is a genuine engagement count) but treated as a
*snapshot* value — see §2 and the leakage note in the final cell.

In [2]:
DROP_COLS = {
    "user_id":       "100% null",
    "year":          "100% null; date is in date_iso/date_unix",
    "num_playlists": "100% null (non-bulk endpoint, excluded by design)",
    "score":         "90.1% null, unclear semantics (sparse rating, not a count)",
    "thumbs_up":     "constant 1 -> zero information",
    "has_meta":      "constant False -> zero information (abandoned metadata pass)",
}
# sanity: confirm the 'no information' claims before dropping
assert nodes["thumbs_up"].nunique() == 1, "thumbs_up not constant - re-check before dropping"
assert nodes["has_meta"].nunique() == 1, "has_meta not constant - re-check before dropping"
for c in ["user_id", "year", "num_playlists"]:
    assert nodes[c].isna().all(), f"{c} is not 100% null - re-check"

clean = nodes.drop(columns=list(DROP_COLS)).copy()
print("dropped:", list(DROP_COLS))
print("remaining columns:", list(clean.columns))

dropped: ['user_id', 'year', 'num_playlists', 'score', 'thumbs_up', 'has_meta']
remaining columns: ['upload_id', 'user_name', 'user_real_name', 'name', 'date_iso', 'date_unix', 'license', 'license_url', 'num_scores', 'num_files', 'n_remixes', 'n_sources', 'usertags', 'systags', 'page_url']


## 2. Fill `num_scores`

`num_scores` is the number of ratings a track received. ccHost omits the key entirely when a track has no ratings, so a null means a true zero. 8.8 % of rows (4,506) are null; we fill them with 0 and record a boolean flag, so the imputation is always visible downstream.

**Discipline note.** `num_scores` is *today's* value, not the value at posting time, so it is a popularity confound. It stays in the file for description and diagnostics only. It is **not** used as a feature in any model; this is stated as a limitation in the thesis.

In [3]:
clean["num_scores_was_null"] = clean["num_scores"].isna()
clean["num_scores"] = clean["num_scores"].fillna(0).astype(int)
print("num_scores nulls filled with 0:", int(clean['num_scores_was_null'].sum()),
      f"({100*clean['num_scores_was_null'].mean():.1f}%)")
print("num_scores range now:", (clean.num_scores.min(), clean.num_scores.max()))

num_scores nulls filled with 0: 4506 (8.8%)
num_scores range now: (0, 251)


## 3. Outcome labels from the lineage graph

**Direction convention.** Errors here fail silently, so the rule is restated every time: a track **was remixed** iff it has *out*-edges (parent → child).

**Labels**
- `is_remixed` — **primary label**: out-degree > 0 in the **local** graph. 18,766 positives (36.45 %).
- `spawned_further_remix` — **secondary label**: a remixed track with at least one local child that is *itself* remixed, i.e. a cascade of two or more generations. 8,104 positives (15.74 %).
- `is_remixed_counter` — the rejected alternative: the site counter `n_remixes` > 0.
- `counter_only_positive` — audit flag for the 1,387 tracks (2.7 %) where the counter says "remixed" but the graph has no child. These are labelled **0** by `is_remixed`.

**Structural columns**
- `out_degree_local` — a current snapshot value, used for description and diagnostics only.
- `in_degree_local` — the number of on-site parents. It is fixed at posting time, because a track's sources are declared at upload, so it is safe for RQ2.

**Deprecated columns: do not use.** `out_degree_all` and `is_remixed_all` count pool edges too. Pool edges point from off-site sample-pool items to on-site tracks, and pool items are never tracks, so these columns cannot add real information about any track's outcome. They are also contaminated: 536 pool edges carry pool-item IDs that numerically equal real track IDs, which wrongly marks 236 tracks as remixed. No local+pool outcome is reported anywhere in the thesis.

In [4]:
local = edges[edges.edge_type == "local"]

out_local = local.groupby("parent_id").size()
out_all   = edges.groupby("parent_id").size()
in_local  = local.groupby("child_id").size()

clean["out_degree_local"] = clean.upload_id.map(out_local).fillna(0).astype(int)
clean["out_degree_all"]   = clean.upload_id.map(out_all).fillna(0).astype(int)
clean["in_degree_local"]  = clean.upload_id.map(in_local).fillna(0).astype(int)

# ---- labels ----
clean["is_remixed"]          = clean.out_degree_local > 0          # PRIMARY
clean["is_remixed_all"]      = clean.out_degree_all   > 0          # robustness
clean["is_remixed_counter"]  = clean.n_remixes        > 0          # rejected alt
clean["counter_only_positive"] = clean.is_remixed_counter & ~clean.is_remixed

# secondary: a local child that is itself remixed
remixed_ids = set(clean.loc[clean.is_remixed, "upload_id"])
spawners    = set(local.loc[local.child_id.isin(remixed_ids), "parent_id"])
clean["spawned_further_remix"] = clean.upload_id.isin(spawners)

print("is_remixed positives          :", int(clean.is_remixed.sum()),
      f"({clean.is_remixed.mean():.4f})")
print("spawned_further_remix positives:", int(clean.spawned_further_remix.sum()),
      f"({clean.spawned_further_remix.mean():.4f})")
print("counter_only_positive (phantom):", int(clean.counter_only_positive.sum()),
      f"({clean.counter_only_positive.mean():.4f})")

is_remixed positives          : 18766 (0.3645)
spawned_further_remix positives: 8104 (0.1574)
counter_only_positive (phantom): 1387 (0.0269)


## 4. Self-checks against `summary.json`

If the edge direction is ever flipped, or the graph is rebuilt differently, these
assertions fail loudly instead of producing a plausible-but-wrong label. This is
the single cheapest guard against the project's biggest silent-failure risk.

In [5]:
exp = summary["outcomes"]["local"]
assert int(clean.is_remixed.sum()) == exp["remixed_at_least_once"], "is_remixed count != summary"
assert abs(clean.is_remixed.mean() - exp["base_rate_remixed_once"]) < 1e-3, "is_remixed rate != summary"
assert int(clean.spawned_further_remix.sum()) == exp["spawned_further_remix"], "spawned count != summary"
assert abs(clean.spawned_further_remix.mean() - exp["base_rate_spawned_further"]) < 1e-3, "spawned rate != summary"
assert clean.upload_id.is_unique, "duplicate upload_id"
# every graph-positive must have a dated child (the whole point of the choice)
assert clean.loc[clean.is_remixed, "out_degree_local"].min() >= 1
print("all self-checks passed")
print("\nlabel agreement (graph vs counter):")
print(pd.crosstab(clean.is_remixed, clean.is_remixed_counter,
                  rownames=['graph is_remixed'], colnames=['counter>0']))

all self-checks passed

label agreement (graph vs counter):
counter>0         False  True 
graph is_remixed              
False             31333   1387
True                  0  18766


## 5. What is safe for RQ2 and what is not (handed on to `03`)

These categories are built into the column set, so `03` can select features without revisiting the decisions.

- **Safe at posting time** (fixed when the track is uploaded): `usertags`, `license`, `license_url`, `name`, `date_iso` / `date_unix`, `num_files`, `n_sources`, `in_degree_local`, and the technical tokens in `systags` (audio format, bitrate).
- **Snapshot values that would leak** (today's value, not the value at posting time): `num_scores`, `n_remixes` and `out_degree_local`. None of these is used as a model feature. `out_degree_all` / `is_remixed_all` are deprecated (see §3).
- **Suspect:** the `editorial_pick` token in `systags` is a curatorial status assigned at an unknown time and may correlate with the outcome, so it is excluded from features.

In [6]:
COL_ORDER = [
    # identity / time
    "upload_id", "user_name", "user_real_name", "name", "date_iso", "date_unix",
    # post-time-safe attributes
    "license", "license_url", "num_files", "n_sources", "usertags", "systags",
    "in_degree_local",
    # snapshot / outcome-ish (RQ1 & robustness only)
    "num_scores", "num_scores_was_null", "n_remixes",
    "out_degree_local", "out_degree_all",
    # labels
    "is_remixed", "spawned_further_remix",
    "is_remixed_all", "is_remixed_counter", "counter_only_positive",
    "page_url",
]
clean = clean[COL_ORDER] 
clean.to_csv(OUT_NODES, index=False)
print("wrote", OUT_NODES, clean.shape)

dd = pd.DataFrame({
    "dtype": clean.dtypes.astype(str),
    "n_null": clean.isna().sum().values,
})
print("\n--- data dictionary ---")
print(dd.to_string())
print("\n--- head ---")
print(clean.head(3).T.to_string())

wrote data/processed/nodes_clean.csv (51486, 24)

--- data dictionary ---
                       dtype  n_null
upload_id              int64       0
user_name                str       0
user_real_name           str       0
name                     str       1
date_iso                 str       0
date_unix              int64       0
license                  str       0
license_url              str       0
num_files              int64       0
n_sources              int64       0
usertags                 str    2849
systags                  str       0
in_degree_local        int32       0
num_scores             int32       0
num_scores_was_null     bool       0
n_remixes              int64       0
out_degree_local       int32       0
out_degree_all         int32       0
is_remixed              bool       0
spawned_further_remix   bool       0
is_remixed_all          bool       0
is_remixed_counter      bool       0
counter_only_positive   bool       0
page_url                 str       0



## 6. For the thesis — limitation paragraph (ready to adapt)

> The remix outcome is derived from the reconstructed lineage graph: a track is
> labelled *remixed* iff it has at least one outgoing edge to a child track present
> in the dataset. This label disagrees with ccMixter's own `n_remixes` counter for
> 2.7 % of tracks (1,387 of 51,486). The disagreement runs in one direction only:
> the counter reports a remix with no corresponding child in the crawl. Only 14 of
> these cases involve pool/off-site links, and only 41 are explained by the 684
> time-inverted edges removed during cleaning. Almost all involve a single remix
> (1,295), which is consistent with stale counters or children that were later
> deleted. We label these tracks *not remixed*, because they cannot be placed in
> time and so cannot support the time-based features used for prediction. The
> resulting label noise is at most 2.7 % and runs in one direction. Excluding the
> affected tracks leaves test performance unchanged (AUC 0.842 → 0.843).

**Robustness check (done in `04`).** The counter cannot serve as an alternative target, because it has no dates and so cannot form horizon labels. The equivalent check, excluding the `counter_only_positive` tracks from training and test, is reported in `04` §7.